# Position-wise Feed Forward

$$
FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

Một mạng feed-forward 2 lớp (Linear → ReLU → Dropout → Linear) áp dụng **độc lập lên từng vị trí (token)** trong chuỗi — cùng một bộ trọng số dùng chung cho mọi vị trí.


In [1]:
import numpy as np

`Parameter` và `Linear` là các khối cơ bản dùng lại từ multi-head attention (xem notebook Multi-head Attention).

In [2]:
class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = None


class Linear:
    def __init__(self, in_features, out_features):
        self.W = Parameter(np.random.randn(in_features, out_features) / np.sqrt(in_features))
        self.b = Parameter(np.zeros((out_features,)))

    def __call__(self, x):
        return np.matmul(x, self.W.data) + self.b.data

`PositionwiseFeedForward`: `linear1` → `relu` → `dropout` → `linear2`. `

In [3]:
class PositionwiseFeedForward:
    def __init__(self, d_model, hidden_units, drop_prop):
        self.linear1 = Linear(d_model, hidden_units)
        self.linear2 = Linear(hidden_units, d_model)
        self.drop_prop = drop_prop
        self.training = True

    def relu(self, x):
        return np.maximum(0, x)

    def dropout(self, x):
        if not self.training or self.drop_prop == 0:
            return x
        mask = (np.random.rand(*x.shape) > self.drop_prop).astype(x.dtype)
        return x * mask / (1 - self.drop_prop)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [4]:
d_model = 512
ffn_hidden = 2048
drop_prob = 0.1
batch_size = 30
max_sequence_length = 200

x = np.random.randn(batch_size, max_sequence_length, d_model)
model = PositionwiseFeedForward(d_model=d_model, hidden_units=ffn_hidden, drop_prop=drop_prob)
out = model.forward(x)
out.shape, type(out)

((30, 200, 512), <class 'numpy.ndarray'>)